In [28]:
import rasterio
from rasterio.merge import merge
import numpy as np
import glob
import os
from scipy.optimize import curve_fit
import pandas as pd
import matplotlib.pyplot as plt

## load in the data
directory_path = 'H:/My Drive/NDVI_Exports/DLCTemp'

## list of file paths for each year's raster
yearly_files = sorted(glob.glob(os.path.join(directory_path, '*.tif')))

## list to store the opened raster files
raster_data = []
shapes = []

for file in yearly_files:
    with rasterio.open(file) as src:
        print(f"Reading file: {file}")
        print(f"Shape: {src.shape}")
        print(f"Number of Bands: {src.count}")
        print(f"CRS: {src.crs}")
        print(f"Transform: {src.transform}")
        print(f"Data type: {src.dtypes[0]}")
        print()

        ## read all bands of the raster data
        data = src.read()  # reading all bands
        raster_data.append(data)
        shapes.append(data.shape)

## find the maximum dimensions
max_shape = np.max([data.shape[1:] for data in raster_data], axis=0)  # rows, cols
print("Max Shape (rows, cols):", max_shape)

## initialize list for all bands combined
combined_raster_data = []

## pad each array to the maximum shape
for idx, data in enumerate(raster_data):
    for band in range(data.shape[0]):
        padded = np.pad(data[band], 
                        ((0, max_shape[0] - data.shape[1]),  # pad rows
                         (0, max_shape[1] - data.shape[2])),  # pad columns
                        mode='constant', constant_values=np.nan)
        combined_raster_data.append(padded)
        print(f"Original Shape for file {idx}, band {band}: {data[band].shape}")
        print(f"Padded Shape for file {idx}, band {band}: {padded.shape}")

## convert to numpy array
stacked_data = np.stack(combined_raster_data, axis=0)

## check shape and data to confirm
print("Stacked Data Shape (total bands, rows, columns):", stacked_data.shape)




Reading file: H:/My Drive/NDVI_Exports/DLCTemp\step7_NDVI_2001.tif
Shape: (457, 881)
Number of Bands: 45
CRS: EPSG:4326
Transform: | 0.00, 0.00,-112.50|
| 0.00,-0.00, 33.26|
| 0.00, 0.00, 1.00|
Data type: float64

Reading file: H:/My Drive/NDVI_Exports/DLCTemp\step7_NDVI_2002.tif
Shape: (457, 881)
Number of Bands: 46
CRS: EPSG:4326
Transform: | 0.00, 0.00,-112.50|
| 0.00,-0.00, 33.26|
| 0.00, 0.00, 1.00|
Data type: float64

Max Shape (rows, cols): [457 881]
Original Shape for file 0, band 0: (457, 881)
Padded Shape for file 0, band 0: (457, 881)
Original Shape for file 0, band 1: (457, 881)
Padded Shape for file 0, band 1: (457, 881)
Original Shape for file 0, band 2: (457, 881)
Padded Shape for file 0, band 2: (457, 881)
Original Shape for file 0, band 3: (457, 881)
Padded Shape for file 0, band 3: (457, 881)
Original Shape for file 0, band 4: (457, 881)
Padded Shape for file 0, band 4: (457, 881)
Original Shape for file 0, band 5: (457, 881)
Padded Shape for file 0, band 5: (457, 881

In [30]:
## calculate min and max along the stack (axis 0)
min_val = np.nanmin(stacked_data, axis=0)
max_975_quantile = np.nanquantile(stacked_data, 0.975, axis=0)

## avoid division by zero by adding a small epsilon value
epsilon = 1e-10
denominator = max_975_quantile - min_val + epsilon

## get the scaled data of the raster stack
scaled_data = (stacked_data - min_val) / denominator

## check the scaled data values to confirm
print("Min Values:", min_val)
print("Max 97.5th Percentile Values:", max_975_quantile)
print("Scaled Data:", scaled_data)

## optional: check the shape of the scaled_data to confirm
print("Scaled Data Shape:", scaled_data.shape) # bands, rows, columns


C:\Users\tlasharr\AppData\Local\Temp\ipykernel_52196\3839273038.py:2: RuntimeWarning: All-NaN slice encountered
  min_val = np.nanmin(stacked_data, axis=0)
c:\Users\tlasharr\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1650: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


Min Values: [[       nan        nan        nan ...        nan        nan        nan]
 [0.11948205 0.11948205 0.11948205 ... 0.12156167 0.12156167 0.12156167]
 [0.11948205 0.11948205 0.11948205 ... 0.12156167 0.12156167 0.12156167]
 ...
 [0.11910711 0.11910711 0.11910711 ... 0.10762884 0.10762884 0.10762884]
 [0.11910711 0.11910711 0.11910711 ... 0.10762884 0.10762884 0.10762884]
 [0.11676218 0.11223988 0.11223988 ... 0.10762884 0.10762884 0.10762884]]
Max 97.5th Percentile Values: [[       nan        nan        nan ...        nan        nan        nan]
 [0.2653441  0.2653441  0.2653441  ... 0.35330725 0.35330725 0.35330725]
 [0.2653441  0.2653441  0.2653441  ... 0.35330725 0.35330725 0.35330725]
 ...
 [0.26076046 0.26076046 0.26076046 ... 0.27557965 0.27557965 0.27557965]
 [0.26076046 0.26076046 0.26076046 ... 0.27557965 0.27557965 0.27557965]
 [0.25905973 0.2524741  0.2524741  ... 0.26900946 0.26900946 0.26900946]]
Scaled Data: [[[       nan        nan        nan ...        nan       

In [36]:

## DLC Model
def dlc_model(time, xmidS, scalS, xmidA, scalA):
    return (1 / (1 + np.exp((xmidS - time) / scalS))) - (1 / (1 + np.exp((xmidA - time) / scalA)))


In [55]:
## define the years corresponding to each layer in the stacked_data
start_year = 2001
end_year = 2002

## define the bounds
dlc_bounds = ([30, 1, 167, 1], [228, 30, 288, 35])

## define the years corresponding to each layer in the stacked_data
years = range(start_year, end_year + 1)


## fit the DLC curve
def fit_dlc(scaled_stacked_data):

    num_bands, num_rows, num_cols = scaled_stacked_data.shape
    dlc_params = np.full((num_rows, num_cols, 4), np.nan)  # Initialize with NaNs
    
    for row in range(num_rows):

        for col in range(num_cols):
            ## extract NDVI data for the pixel across all bands
            ndvi_values = scaled_stacked_data[:, row, col]
            
            ## generate time points corresponding to each band
            time = np.arange(1, num_bands + 1)

            ## remove NaNs and infinities from the data
            valid_mask = np.isfinite(ndvi_values)

            ## if there are less than 4 images, don't try to fit the curves
            if np.sum(valid_mask) < 4:
                continue
            
            try:
                ## fit the curves, with bounds on the parameters
                popt, _ = curve_fit(dlc_model, time[valid_mask], ndvi_values[valid_mask], bounds=dlc_bounds)
                dlc_params[row, col, :] = popt

            except Exception as e:
                print(f"Error fitting pixel ({row}, {col}): {e}")
                continue

    return dlc_params

## fit DLC curves
dlc_params = fit_dlc(scaled_data)

## look at the general shape (rows, columns, )
print("DLC Parameters Shape:", dlc_params.shape)


DLC Parameters Shape: (457, 881, 4)


In [54]:
import numpy as np
from scipy.special import expit
import matplotlib.pyplot as plt
import pandas as pd

def fit_dlc_per_pixel(dlc_params, year, interval_days):
    num_rows, num_cols, num_params = dlc_params.shape
    assert num_params == 4, "dlc_params should have 4 parameters per pixel."
    
    n_days = 366 if year in [2004, 2008, 2012, 2016, 2020, 2024] else 365
    time = np.arange(1, n_days + 1, interval_days)
    
    fitted_vals = np.full((num_rows, num_cols, len(time)), np.nan)
    
    for row in range(num_rows):
        for col in range(num_cols):
            params = dlc_params[row, col]
            if np.isnan(params).any():
                continue
            try:
                fitted_curve = dlc_model(time, *params)
                fitted_vals[row, col, :] = fitted_curve
            except Exception as e:
                print(f"Error fitting pixel ({row}, {col}): {e}")
    
    return fitted_vals

## Example usage
year1 = 2001
year2 = 2002

# Assuming stacked_data is a 3D array (total layers, rows, columns)
num_layers = stacked_data.shape[0] // 2  # Assuming 2 years
stacked_data_year1 = stacked_data[:num_layers, :, :]
stacked_data_year2 = stacked_data[num_layers:, :, :]

# DLC parameters for each year
dlc_params_year1 = dlc_params[:, :, :4]  # Adjust as necessary for your data
dlc_params_year2 = dlc_params[:, :, 4:]  # Adjust as necessary for your data

# Fit the DLC model for each year separately
interval_days = 8  # 8-day intervals
fitted_vals_year1 = fit_dlc_per_pixel(dlc_params_year1, year1, interval_days)
fitted_vals_year2 = fit_dlc_per_pixel(dlc_params_year2, year2, interval_days)

## Visualizing the fitted curve and raw data for a specific pixel
def plot_fitted_and_raw(fitted_vals, raw_data, row, col, title, dates):
    plt.figure(figsize=(10, 6))
    plt.plot(dates, raw_data[:, row, col], label='Raw NDVI Data', marker='o', linestyle='None')
    plt.plot(dates, fitted_vals[row, col, :], label='Fitted NDVI Curve', marker='o')
    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel('NDVI')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.show()

# Create dates array for the raw data based on 8-day intervals
start_date_year1 = pd.to_datetime(f'{year1}-01-01')
start_date_year2 = pd.to_datetime(f'{year2}-01-01')
dates_year1 = pd.date_range(start=start_date_year1, periods=stacked_data_year1.shape[0], freq='8D')
dates_year2 = pd.date_range(start=start_date_year2, periods=stacked_data_year2.shape[0], freq='8D')

## Example: row and column indices for the pixel you want to analyze
row_index = 100  # Change this to the specific row index of the pixel
col_index = 150  # Change this to the specific column index of the pixel

# Plot the fitted curve and raw data for a specific pixel (example: row 100, col 150) for year 1
plot_fitted_and_raw(fitted_vals_year1, stacked_data_year1, row_index, col_index, f"Fitted and Raw NDVI Curve for Pixel (100, 150) - {year1}", dates_year1)

# Plot the fitted curve and raw data for a specific pixel (example: row 100, col 150) for year 2
plot_fitted_and_raw(fitted_vals_year2, stacked_data_year2, row_index, col_index, f"Fitted and Raw NDVI Curve for Pixel (100, 150) - {year2}", dates_year2)





AssertionError: dlc_params should have 4 parameters per pixel.